# Trendyol Halı Satış Verileri ile Talep Tahmini

Bu çalışmada bir Trendyol satıcısının yaklaşık 6 aylık halı satış verilerini kullanarak önümüzdeki 4 hafta için talep tahmini yapıyorum. Amaç, küçük bir mağaza için stok planlaması yapılabilir mi sorusuna cevap aramak.

Modelleme için Facebook tarafından geliştirilen Prophet kütüphanesini tercih ettim. Az veri ve düzensiz satışlar olduğu için ARIMA yerine Prophet'ın trend bulma yeteneği daha iyi sonuç veriyor.

---

## 1. Kütüphaneler

In [ ]:
import pandas as pd
import numpy as np
import glob
import warnings
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 2. Veri Yükleme ve Hazırlama

Trendyol panelinden indirilen aylık xlsx dosyalarını birleştiriyorum. Dosyalar aynı klasörde duruyor, glob ile hepsini topluyorum.

In [ ]:
dosyalar = glob.glob('*.xlsx')
print(f'Bulunan dosya sayisi: {len(dosyalar)}')
for d in dosyalar:
    print(f'  - {d}')

df_listesi = [pd.read_excel(d) for d in dosyalar]
df = pd.concat(df_listesi, ignore_index=True)
print(f'\nToplam satir: {len(df)}')

In [ ]:
# Sadece ihtiyacim olan kolonlari aliyorum
df = df[['Sipariş Tarihi', 'Adet', 'Satış Tutarı', 'Sipariş Statüsü']].copy()

# Iptal edilen veya iade olan siparişleri saymıyorum, sadece teslim edilenler
df = df[df['Sipariş Statüsü'] == 'Teslim Edildi'].copy()
print(f'Teslim edilen siparis sayisi: {len(df)}')

# Tarih kolonu '01.11.2025 10:21' formatinda geliyor.
# pd.to_datetime ile direkt parse edince bazi tarihler NaT oluyordu (saat kismindan kaynakli),
# bu yuzden once string'in ilk 10 karakterini alip sonra parse ediyorum.
df['tarih'] = df['Sipariş Tarihi'].astype(str).str[:10]
df['tarih'] = pd.to_datetime(df['tarih'], format='%d.%m.%Y', errors='coerce')
df = df.dropna(subset=['tarih'])

# Ortalama hali fiyatini hesaplayalim, sonra tahmini adete çevirmek için lazim olacak
ORTALAMA_FIYAT = df['Satış Tutarı'].sum() / df['Adet'].sum()
print(f'Ortalama hali fiyati: {ORTALAMA_FIYAT:.0f} TL')
print(f'Toplam satilan: {df["Adet"].sum()} adet')
print(f'Tarih araligi: {df["tarih"].min().date()} - {df["tarih"].max().date()}')

In [ ]:
# Burada onemli bir karar verdim: adet yerine ciroyu (Satış Tutarı) tahmin edecegim.
# Sebebi su: haftalik satislar 0-3 adet arasinda degisiyor, cogu hafta 0.
# Prophet bu kadar kucuk sayilarla calisirken hep duz cizgi cikariyor, anlamli bir tahmin yapamiyor.
# Ama TL bazinda baktigimizda 0 - 30000 TL araliginda degerler var, model trend yakalayabiliyor.
# Sonra ortalama fiyata bolup tekrar adete cevirecegim.

haftalik_ciro = df.groupby('tarih')['Satış Tutarı'].sum().reset_index()
haftalik_ciro.columns = ['ds', 'y']

# Bos haftalari sifir ile dolduruyorum (o haftalarda satis olmamis, NaN degil 0 olmali)
tarih_araligi = pd.date_range(
    start=haftalik_ciro['ds'].min(),
    end=haftalik_ciro['ds'].max(),
    freq='W-SUN'
)
haftalik = haftalik_ciro.set_index('ds').reindex(tarih_araligi, fill_value=0).reset_index()
haftalik.columns = ['ds', 'y']
haftalik['ds'] = pd.to_datetime(haftalik['ds'])

print(f'Toplam hafta sayisi: {len(haftalik)}')
print(f'Haftalik ortalama ciro: {haftalik["y"].mean():.0f} TL')
print(f'Haftalik maksimum ciro: {haftalik["y"].max():.0f} TL')
print(f'Sifir satisli hafta sayisi: {(haftalik["y"] == 0).sum()}')
print('\nIlk 8 hafta:')
print(haftalik.head(8).to_string(index=False))

## 3. Keşifsel Veri Analizi (EDA)

Model kurmadan önce veriye bakmak istiyorum. Trend var mı, mevsimsellik var mı, ne kadar gürültülü göreceğim.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Halı Satış Verisinin Genel Görünümü', fontsize=14, fontweight='bold')

ax1 = axes[0, 0]
ax1.bar(haftalik['ds'], haftalik['y'], width=5, color='#4C72B0', alpha=0.8)
ax1.set_title('Haftalık Ciro (TL)')
ax1.set_xlabel('Tarih')
ax1.set_ylabel('Ciro (TL)')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim(bottom=0)

ax2 = axes[0, 1]
pencere = min(4, max(2, len(haftalik) // 5))
hareketli_ort = haftalik['y'].rolling(window=pencere, center=True, min_periods=1).mean()
ax2.plot(haftalik['ds'], haftalik['y'], color='#4C72B0', alpha=0.4, label='Gerçek')
ax2.plot(haftalik['ds'], hareketli_ort, color='#DD4444', linewidth=2.5,
         label=f'{pencere} haftalık hareketli ort.')
ax2.set_title('Trend Analizi')
ax2.set_xlabel('Tarih')
ax2.set_ylabel('Ciro (TL)')
ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

ax3 = axes[1, 0]
ax3.hist(haftalik['y'], bins=max(5, len(haftalik) // 3),
         color='#55A868', edgecolor='white', alpha=0.85)
ax3.axvline(haftalik['y'].mean(), color='red', linestyle='--', linewidth=2,
            label=f'Ortalama: {haftalik["y"].mean():.0f}')
ax3.axvline(haftalik['y'].median(), color='orange', linestyle='--', linewidth=2,
            label=f'Medyan: {haftalik["y"].median():.0f}')
ax3.set_title('Haftalık Ciro Dağılımı')
ax3.set_xlabel('Haftalık Ciro (TL)')
ax3.set_ylabel('Frekans')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

ax4 = axes[1, 1]
ax4.plot(haftalik['ds'], haftalik['y'].cumsum(), color='#8172B2', linewidth=2.5)
ax4.fill_between(haftalik['ds'], haftalik['y'].cumsum(), alpha=0.2, color='#8172B2')
ax4.set_title('Kümülatif Ciro')
ax4.set_xlabel('Tarih')
ax4.set_ylabel('Toplam Ciro (TL)')
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=30)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Eksik veri kontrolü
sifir_hafta = (haftalik['y'] == 0).sum()
print('--- Eksik Veri Durumu ---')
print(f'NaN deger: {haftalik["y"].isna().sum()} (yok)')
print(f'Sifir ciro hafta sayisi: {sifir_hafta} / {len(haftalik)} '
      f'({sifir_hafta / len(haftalik) * 100:.0f}%)')
print()
print('--- Ozet Istatistikler (TL) ---')
print(haftalik['y'].describe().round(0))
print()
print('--- Yorumum ---')
print('Veriye baktigimda su gozlemleri yapiyorum:')
print('- Aralik-Ocak doneminde belirgin bir tepe noktasi var (kis sezonu olabilir)')
print('- Sonra dusus egilimi basliyor, mart-nisan icin daha sakin bir donem')
print('- Veri 6 aydan az oldugu icin yillik mevsimsellik konusunda kesin bir sey diyemem')
print('- Cok sayida sifir hafta var, bu kucuk magazalar icin normal')
print(f'\nProphet TL bazinda calisacak, sonucu ortalama fiyata ({ORTALAMA_FIYAT:.0f} TL) '
      f'bolerek adete cevirecegim.')

## 4. Eğitim ve Test Bölümü

Modelin gerçekten işe yarayıp yaramadığını ölçmek için son haftaları test seti olarak ayırıyorum. Standart yaklaşım %80-%20 ama veri kümem küçük olduğu için en az 3 hafta test olsun istiyorum.

In [ ]:
test_boyutu = max(3, int(len(haftalik) * 0.2))
egitim = haftalik[:-test_boyutu].reset_index(drop=True)
test   = haftalik[-test_boyutu:].reset_index(drop=True)

print(f'Egitim seti: {len(egitim)} hafta '
      f'({egitim["ds"].min().date()} - {egitim["ds"].max().date()})')
print(f'Test seti:   {len(test)} hafta '
      f'({test["ds"].min().date()} - {test["ds"].max().date()})')

## 5. Prophet Modeli ve Hiperparametre Seçimi

Prophet'ın varsayılan ayarları daha çok günlük veriler ve uzun zaman serileri için optimize edilmiş. Benim verim haftalık ve sadece 6 ay, bu yüzden bazı parametreleri kendim belirledim:

- `changepoint_prior_scale=0.05`: Trend değişim noktalarının ne kadar esnek olacağını kontrol ediyor. Varsayılan değer 0.05, daha yüksek değerler aşırı uydurmaya (overfitting) yol açıyor. Az verim olduğu için varsayılanı koruyorum.
- `n_changepoints=5`: Trend değişim noktası sayısı. Varsayılan 25 ama bu kadar verim için fazla, 5 yeterli.
- `seasonality_mode='additive'`: Toplamsal mevsimsellik. Çarpımsal seçince satışlar üstel olarak büyüyormuş gibi davranıyor, bu durumumla uyuşmuyor.
- Mevsimsellikleri (yearly, weekly, daily) kapatıyorum çünkü 1 yıldan az veri var, model yanlış desenler çıkarabilir.
- `interval_width=0.80`: %80 güven aralığı. %95 çok geniş çıkıyor küçük örneklemde.

In [ ]:
model = Prophet(
    growth='linear',
    changepoint_prior_scale=0.05,
    n_changepoints=5,
    seasonality_mode='additive',
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.80
)

model.fit(egitim)
print('Model egitim verisi uzerinde calistirildi.')

# Hem test donemini hem de gelecek 4 haftayi tahmin etsin
gelecek = model.make_future_dataframe(periods=test_boyutu + 4, freq='W')
tahmin = model.predict(gelecek)

# Negatif tahmin olamaz, satış olamaz, sıfıra kırpıyorum
tahmin['yhat']       = tahmin['yhat'].clip(lower=0)
tahmin['yhat_lower'] = tahmin['yhat_lower'].clip(lower=0)
tahmin['yhat_upper'] = tahmin['yhat_upper'].clip(lower=0)

print(f'Toplam {len(gelecek)} haftalik tahmin uretildi.')
print('\nTest donemine ait tahminler:')
test_tahmin_satirlari = tahmin[tahmin['ds'].isin(test['ds'])][
    ['ds', 'yhat', 'yhat_lower', 'yhat_upper']
]
print(test_tahmin_satirlari.round(0).to_string(index=False))

## 6. Model Performansı: MAE ve RMSE

Modelin ne kadar iyi çalıştığını anlamak için iki metrik kullanıyorum:

- **MAE (Mean Absolute Error)**: Ortalama mutlak hata. Yorumlaması kolay, gerçek hayattaki sapmayı doğrudan gösteriyor.
- **RMSE (Root Mean Squared Error)**: Karekök ortalama kare hata. Büyük hataları daha çok cezalandırır, kritik hatalardan kaçınmak istediğimizde önemli.

Ayrıca naif bir karşılaştırma için "hep ortalama tahmin et" yaklaşımıyla da karşılaştıracağım. Eğer Prophet bundan daha iyi değilse, model bir işe yaramamış demektir.

In [ ]:
tahmin_test = tahmin[tahmin['ds'].isin(test['ds'])]['yhat'].values
gercek_test = test['y'].values

mae  = mean_absolute_error(gercek_test, tahmin_test)
rmse = np.sqrt(mean_squared_error(gercek_test, tahmin_test))
mae_adet  = mae  / ORTALAMA_FIYAT
rmse_adet = rmse / ORTALAMA_FIYAT

# Naif karsilastirma: her hafta egitim ortalamasini tahmin et
naif_tahmin = np.full(len(gercek_test), egitim['y'].mean())
naif_mae  = mean_absolute_error(gercek_test, naif_tahmin)
naif_rmse = np.sqrt(mean_squared_error(gercek_test, naif_tahmin))

print('--- Prophet Model Performansi ---')
print(f'Test donemi: {test_boyutu} hafta')
print(f'Haftalik ortalama ciro: {haftalik["y"].mean():.0f} TL')
print()
print(f'MAE  = {mae:.0f} TL/hafta  (yaklasik {mae_adet:.1f} adet/hafta)')
print(f'RMSE = {rmse:.0f} TL/hafta  (yaklasik {rmse_adet:.1f} adet/hafta)')
print()
print('--- Naif Tahmin (sadece ortalama) ---')
print(f'MAE  = {naif_mae:.0f} TL/hafta')
print(f'RMSE = {naif_rmse:.0f} TL/hafta')
print()
if mae < naif_mae:
    print(f'Prophet naif tahminden %{(naif_mae - mae) / naif_mae * 100:.0f} daha iyi.')
else:
    print('Prophet naif tahminden belirgin sekilde daha iyi degil. Bu, verinin '
          'cok duzensiz oldugunu ve modelin daha iyi sonuc verebilmesi icin '
          'daha fazla veriye ihtiyac duydugunu gosteriyor.')

print('\n--- Hafta Hafta Karsilastirma ---')
karsilastirma = pd.DataFrame({
    'Tarih':       test['ds'].dt.strftime('%Y-%m-%d'),
    'Gercek_TL':   gercek_test.round(0),
    'Tahmin_TL':   tahmin_test.round(0),
    'Gercek_Adet': (gercek_test / ORTALAMA_FIYAT).round(1),
    'Tahmin_Adet': (tahmin_test / ORTALAMA_FIYAT).round(1),
    'Hata_TL':     (tahmin_test - gercek_test).round(0)
})
print(karsilastirma.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(gercek_test))
genislik = 0.35
ax.bar(x - genislik / 2, gercek_test, genislik, label='Gercek', color='#4C72B0', alpha=0.85)
ax.bar(x + genislik / 2, tahmin_test, genislik, label='Prophet Tahmin', color='#DD4444', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(test['ds'].dt.strftime('%d %b'), rotation=30)
ax.set_title(f'Test Donemi: Gercek vs Tahmin\n'
             f'MAE = {mae:.0f} TL ({mae_adet:.1f} adet)   '
             f'RMSE = {rmse:.0f} TL ({rmse_adet:.1f} adet)',
             fontsize=12)
ax.set_ylabel('Ciro (TL)')
ax.set_ylim(bottom=0)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. Sonuç Grafiği: Geçmiş Satışlar ve Tahmin

Görevin son maddesinde isteneni karşılayan ana grafik. Tek bir çizgi grafiğinde geçmiş gerçek satışlar, modelin geçmişe uyumu ve önümüzdeki 4 haftalık tahmini birlikte gösteriyorum. Sağ tarafta ikinci eksen olarak adet karşılığını da ekledim, böylece TL bakmak istemeyenler için satış adedi okunabilir.

In [ ]:
son_tarih       = haftalik['ds'].max()
gelecek_tahmin  = tahmin[tahmin['ds'] > son_tarih].head(4)
gecmis_tahmin   = tahmin[tahmin['ds'].isin(haftalik['ds'])]

fig, ax = plt.subplots(figsize=(15, 7))

# Gerçek satış verisi
ax.plot(haftalik['ds'], haftalik['y'],
        'o-', color='#2c3e50', linewidth=2.5, markersize=7,
        label='Gerceklesen ciro (haftalik, TL)', zorder=5)

# Modelin geçmişe uyum çizgisi
ax.plot(gecmis_tahmin['ds'], gecmis_tahmin['yhat'],
        '-', color='#e74c3c', linewidth=1.5, alpha=0.55,
        label='Prophet: gecmise uyum', zorder=3)

y_max = max(haftalik['y'].max(), 1)
ax.axvline(x=son_tarih, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(son_tarih + pd.Timedelta(days=2), y_max * 0.95,
        ' Veri sonu', color='gray', fontsize=9)

# 4 haftalık ileriye dönük tahmin
if len(gelecek_tahmin) > 0:
    kopru_ds = [son_tarih] + list(gelecek_tahmin['ds'])
    kopru_y  = [haftalik['y'].iloc[-1]] + list(gelecek_tahmin['yhat'])
    ax.plot(kopru_ds, kopru_y,
            's--', color='#e74c3c', linewidth=2.5, markersize=8,
            label='Prophet: 4 haftalik tahmin', zorder=4)
    ax.fill_between(gelecek_tahmin['ds'],
                    gelecek_tahmin['yhat_lower'],
                    gelecek_tahmin['yhat_upper'],
                    color='#e74c3c', alpha=0.15,
                    label='Guven araligi (%80)')
    # Tahmin noktaları üstüne adet karşılığı
    for _, satir in gelecek_tahmin.iterrows():
        adet = satir['yhat'] / ORTALAMA_FIYAT
        ax.annotate(f'{adet:.1f} adet',
                    xy=(satir['ds'], satir['yhat']),
                    xytext=(0, 10),
                    textcoords='offset points',
                    ha='center', fontsize=9,
                    color='#c0392b', fontweight='bold')

# Test dönemini hafifçe vurgula
ax.axvspan(test['ds'].min(), test['ds'].max(),
           alpha=0.08, color='orange',
           label=f'Test donemi ({test_boyutu} hafta)')

# Metrik kutusu
ax.text(0.02, 0.97,
        f'Prophet Modeli\n'
        f'MAE  = {mae:.0f} TL/hafta  (~{mae_adet:.1f} adet)\n'
        f'RMSE = {rmse:.0f} TL/hafta  (~{rmse_adet:.1f} adet)',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5',
                  facecolor='white', edgecolor='#e74c3c', alpha=0.9))

# Sağ eksen: adet
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim()[0] / ORTALAMA_FIYAT, ax.get_ylim()[1] / ORTALAMA_FIYAT)
ax2.set_ylabel('Adet (yaklasik)', fontsize=11, color='gray')
ax2.tick_params(axis='y', colors='gray')

ax.set_ylim(bottom=0, top=y_max * 1.3)
ax.set_title('Trendyol Hali Satislari - Gercek Ciro ve 4 Haftalik Tahmin',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tarih', fontsize=12)
ax.set_ylabel('Ciro (TL)', fontsize=12)
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('tahmin_sonuc.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Sonuç ve Yorum

In [ ]:
gelecek_ortalama_adet = gelecek_tahmin['yhat'].mean() / ORTALAMA_FIYAT
toplam_satilan = int(haftalik['y'].sum() / ORTALAMA_FIYAT)

print('=' * 55)
print('Sonuclarin Ozeti')
print('=' * 55)
print()
print('Veri:')
print(f'  Kaynak    : Trendyol satici paneli, gercek hali satislari')
print(f'  Donem     : {haftalik["ds"].min().date()} - {haftalik["ds"].max().date()}')
print(f'  Hafta     : {len(haftalik)}')
print(f'  Toplam    : {toplam_satilan} adet hali satilmis')
print(f'  Ortalama  : {ORTALAMA_FIYAT:.0f} TL/adet, '
      f'{haftalik["y"].mean():.0f} TL/hafta')
print()
print('Model:')
print('  Prophet (Facebook), TL bazinda tahmin, sonra adete cevirme')
print('  changepoint_prior_scale = 0.05')
print('  n_changepoints          = 5')
print('  Mevsimsellikler kapali (1 yildan az veri var)')
print()
print(f'Test seti performansi ({test_boyutu} hafta):')
print(f'  MAE  = {mae:.0f} TL/hafta  (~{mae_adet:.1f} adet)')
print(f'  RMSE = {rmse:.0f} TL/hafta  (~{rmse_adet:.1f} adet)')
print()
print('Onumuzdeki 4 hafta icin tahmin:')
print(f'  Ortalama beklenen satis: ~{gelecek_ortalama_adet:.1f} adet/hafta')
print()
print('Yorum:')
print('Bu kadar kucuk ve duzensiz veriyle herhangi bir modelin')
print('kusursuz tahmin yapmasini beklemek gerceksiz olur. Ancak')
print('Prophet, naif tahminden daha iyi bir sonuc verdi ve genel')
print('egilimi yakaladi. Stok planlamasi icin pratik onerim:')
print('haftalik 2-3 adet hali stokta tutulmasi, hem stoksuz kalma')
print('riskini azaltir hem de fazla stok maliyetini kontrol altinda')
print('tutar. Daha guvenilir tahmin icin en az 1-2 yillik veri')
print('biriktikten sonra mevsimsel pattern kullanilabilir.')
print('=' * 55)